# What is CDC in Databricks?
Change Data Capture (CDC) means capturing:
- INSERT
- UPDATE
- DELETE

**From a source system and applying them incrementally to a Delta table.**

### CDC Implementation Patterns in Databricks
There are 4 real-world patterns:
1. MERGE INTO (Most common)
2. Change Data Feed (CDF) – Delta feature
3. Streaming CDC (Auto Loader + MERGE)
4. Timestamp / Version-based CDC

### Pattern 1: CDC Using MERGE INTO (Most Used)
When to use
* Source sends full rows
* Or source sends operation flag (I/U/D)

In [0]:
%sql
-- Step 1: Target Delta Table (Silver)
CREATE TABLE db_dlt_proj.silver.employee (
  emp_id INT,
    name STRING,
    department STRING,
    salary INT,
    is_active BOOLEAN,
    updated_at TIMESTAMP
)
USING DELTA;

***Step 2: Incoming CDC Data (Bronze)***
| emp_id | name    | dept    | salary | op | updated_at |
| ------ | ------- | ------- | ------ | -- | ---------- |
| 1      | Shubham | IT      | 75000  | U  | 2025-01-08 |
| 2      | Amit    | HR      | 60000  | D  | 2025-01-08 |
| 3      | Neha    | Finance | 90000  | I  | 2025-01-08 |


SQL version of merge operation

In [0]:
%sql
-- Step 3: MERGE Logic
MERGE INTO db_dlt_proj.silver.employee AS t
USING db_dlt_poj.bronze.employee_cdc AS s
ON t.emp_id = s.emp_id
WHEN MATCHED AND s.OP = "U" THEN
  UPDATE SET
    t.name = s.name,
    t.department = s.department,
    t.salary = s.salary,
    t.updated_at = s.updated_at
WHEN MATCHED AND s.op = "D" THEN
  UPDATE SET 
    t.is_active = false,
    t.updated_at = s.updated_at
WHEN NOT MATCHED AND s.OP = "I" THEN
  INSERT (
    emp_id, name, department, salary, is_active, updated_at
  )
  VALUES (
    s.emp_id, s.name, s.dept, s.salary, true, s.updated_at
  );

Pyspark version of merge operation

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

# Load target (silver) Delta table
target_df = DeltaTable.forName(spark, "db_dlt_poj.silver.employee")
# Load source (bronze CDC) table
source_df = spark.table("db_dlt_poj.bronze.employee_cdc")
(
    target_df.alias("t")
    .merge(
        source_df.alias("s"),
        "t.emp_id = s.emp_id"
    )
    
    # UPDATE for UPDATE operation
    .whenMatchedUpdate(
        condition="s.op = 'U'",
        set={
            "name": "s.name",
            "department": "s.department",
            "salary": "s.salary",
            "updated_at": "s.updated_at"
        }
    )
    
    # UPDATE for DELETE operation (soft delete)
    .whenMatchedUpdate(
        condition="s.op = 'D'",
        set={
            "is_active": "false",
            "updated_at": "s.updated_at"
        }
    )
    
    # INSERT for INSERT operation
    .whenNotMatchedInsert(
        condition="s.op = 'I'",
        values={
            "emp_id": "s.emp_id",
            "name": "s.name",
            "department": "s.dept",
            "salary": "s.salary",
            "is_active": "true",
            "updated_at": "s.updated_at"
        }
    )
    .execute()
)


### Pattern 2: CDC Using Delta Change Data Feed (CDF) ⭐

When to use:
* Delta → Delta CDC
* Audit / downstream sync
* Gold layer builds

**Step 1: Enable CDF**

In [0]:
%sql
-- Note: Must be enabled BEFORE changes happen
ALTER TABLE db_dlt_proj.bronze.employee_cdc
SET TBLPROPERTIES (
  delta.enablechangedataFeed = TRUE
)

**Step 2: Read CDC Changes**

**_Version-based (Most reliable)_**

Stored versions (control_table)

In [0]:
%sql
CREATE TABLE db_dlt_proj.silver.cdc_watermark (
  table_name STRING,
  last_version BIGINT,
  last_updated TIMESTAMP
)
USING DELTA;

In [0]:
last_version = spark.read \
  .table("db_dlt_proj.bronze.cdc_watermark") \
  .filter(col("table_name") == "db_dlt_proj.bronze.employee_cdc") \
  .select("last_version") \
  .collect()[0]["last_version"]

df_cdc = spark.read \
  .format("delta") \
  .option("readChangeFeed", "true") \
  .option("startingVersion", last_version) \
  .table("db_dlt_proj.bronze.employee_cdc")

# UPDATE control table version after data merge into target

new_last_version = df_cdc.agg(
    max("_commit_version").alias("max_version")
).collect()[0]["max_version"]

spark.sql(f"""
MERGE INTO control.cdc_watermark t
USING (
  SELECT
    'db_dlt_proj.bronze.employee_cdc' AS table_name,
    {new_last_version} AS last_version,
    current_timestamp() AS last_updated
) s
ON t.table_name = s.table_name
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
""")


**CDC Metadata Columns**
| Column              | Meaning                                              |
| ------------------- | ---------------------------------------------------- |
| `_change_type`      | insert / update_preimage / update_postimage / delete |
| `_commit_version`   | Delta version                                        |
| `_commit_timestamp` | Commit time                                          |


**Step 3: Filter Only Latest Changes**

In [0]:
df_cdc.filter(col("_change_type").isin("insert","update_porstimage","delete"))

**Step 4: Apply Changes to TARGET Delta**

In [0]:
%sql
MERGE INTO target.employee t
USING changes_view s
ON t.emp_id = s.emp_id

WHEN MATCHED AND s._change_type = 'update_postimage' THEN UPDATE SET *
WHEN MATCHED AND s._change_type = 'delete' THEN UPDATE SET is_active = false
WHEN NOT MATCHED AND s._change_type = 'insert' THEN INSERT *;

### Pattern 3: Streaming CDC (Auto Loader + MERGE)

**Step 1: Stream CDC Data**

In [0]:
df_stream = spark.readStream.format("cloudFiles")\
                            .option("cloudFiles.format","json")
                            .option("cloudFiles.schemaLocation","dbfs:/FileStore/tables/schema_location")\
                            .option("cloudFiles.schemaEvolutionMode","rescue")\
                            .option("cloudFiles.maxFilesPerTrigger",1)\
                            .option("cloudFiles.useNotifications",True)\
                            .load("/mmt/cdc/input")

**Step 2: foreachBatch MERGE**

In [0]:
def merge_cdc(microBatchDF, batchId):
  microBatchDF.createOrReplaceTempView("cdc_batch")

    spark.sql("""
      MERGE INTO silver.employee t
      USING cdc_batch s
      ON t.emp_id = s.emp_id

      WHEN MATCHED AND s.op = 'U' THEN UPDATE SET *
      WHEN MATCHED AND s.op = 'D' THEN UPDATE SET is_active = false
      WHEN NOT MATCHED AND s.op = 'I' THEN INSERT *
    """)

df_stream.writeStream\
         .foreachBatch(merge_cdc)\
           option("checkpointLocation","dbfs:/FileStore/tables/checkpoint")\
           .start()

### Streaming CDC (Auto Managed)

In [0]:
# when your source is a streaming table then streaming cdc can handle autommatically
df_stream = spark.readStream \
  .format("delta") \
  .option("readChangeFeed", "true") \
  .table("source.employee")

### Pattern 4: Timestamp-Based CDC (Simple Sources)
Use case
* No CDC flags
* Only `last_updated_ts`

In [0]:
last_ts = spark.read.tale("control.cdc_watermark").collect()[0]['last_ts']
df_delta = spark.read.table("db_dlt_proj.bronze.employee")\
                      .filter(col("update_at") > last_ts)

# Then apply MERGE

### CDC Design Best Practices (VERY IMPORTANT)

- ✔ Always land raw data in Bronze
- ✔ Apply CDC in Silver
- ✔ Use surrogate keys if source key is unstable
- ✔ Partition Delta table properly
- ✔ Vacuum carefully (CDC depends on history)

# BEST OPTION: Delta Change Data Feed (CDF) ⭐⭐⭐⭐⭐

If both source and target are Delta tables → CDF is the BEST and CLEANEST solution.
| Reason                     | Explanation                                    |
| -------------------------- | ---------------------------------------------- |
| True incremental           | Reads only changed rows (insert/update/delete) |
| No MERGE scanning          | Avoids full table scans                        |
| Built-in metadata          | `_change_type`, `_commit_version`              |
| Exactly-once               | Delta guarantees consistency                   |
| Designed for Delta → Delta | Native feature                                 |

**Source Delta (CDF enabled) --> Read incremental changes using CDF --> Apply to Target Delta (MERGE or append)**